# 40. The Port Capacity & Expansion Timing Problem
## Tier 2: The Classic Heuristic (Variable Neighborhood Descent)

### Goal
Implement Variable Neighborhood Descent (VND) to systematically explore different capacity expansion configurations by iteratively improving solutions through structured neighborhood changes.

### Key assumptions
- Multiple neighborhood structures represent different decision modifications
- Local search can escape local optima by cycling through neighborhoods
- Each neighborhood focuses on specific aspects (timing, capacity, technology, phasing)
- Solution evaluation considers investment, operational, and congestion costs

### Approach (step-by-step)
1. **Initial Solution**: Start with baseline expansion configuration
2. **Neighborhood Structures**: Define 4 neighborhood types for different modifications
3. **Local Search**: Iteratively explore and improve within each neighborhood
4. **Acceptance Criteria**: Accept improvements and restart from first neighborhood
5. **Termination**: Stop when no improvement found in any neighborhood
6. **Solution Analysis**: Evaluate final solution and compare with baseline

### What to look for in the results
- Progressive improvement through neighborhood exploration
- Trade-offs between different solution components
- Convergence to locally optimal solution
- Comparison with initial configuration performance

### Concrete example (from the source)
Port Meridian optimization starting with:
- Expansion year: 2026
- Capacity increase: 40% (+1.28M TEU)
- Technology: Semi-automated
- Phased: False
- Initial cost: $467M

VND explores timing adjustments, capacity level changes, technology selections, and phasing decisions to find improved configuration.

### Why this Tier exists vs Tier 1
Tier 1 provides the theoretical mathematical foundation, but VND offers a practical heuristic approach that can handle larger problem instances and more complex constraints. VND is computationally efficient and provides good solutions quickly, making it suitable for real-world decision-making where exact optimization may be too slow or complex.

### Pros / Cons vs Tier 1
**Pros:**
- Much faster computation time
- Can handle more complex constraints and realistic scenarios
- Provides good solutions quickly for practical decision-making
- Less sensitive to probability estimation errors

**Cons:**
- No guarantee of optimality
- Solution quality depends on neighborhood design
- May get stuck in local optima
- Requires careful parameter tuning

### When to use this Tier
- Medium to large-scale capacity planning problems
- Time-constrained decision environments
- Problems with complex constraints not easily modeled in MDP
- When good solutions are needed quickly rather than optimal ones

In [1]:
# Import required libraries for VND heuristic
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import random
from dataclasses import dataclass, field
from typing import Dict, List, Tuple
import pandas as pd
from copy import deepcopy

# Set random seed for reproducibility
np.random.seed(42)
random.seed(42)

# Set style for better visualizations
plt.style.use('default')
sns.set_palette("husl")

print("Libraries imported successfully for VND heuristic")

Libraries imported successfully for VND heuristic


Libraries imported successfully for VND heuristic


Libraries imported successfully for VND heuristic


Libraries imported successfully for VND heuristic


Libraries imported successfully for VND heuristic


In [ ]:
@dataclass
class ExpansionSolution:
    """Represents a port capacity expansion solution"""
    expansion_year: int  # Year to start expansion (1-10 years from now)
    capacity_increase: float  # Percentage increase (0.15, 0.40, 0.85)
    technology: str  # 'manual', 'semi_automated', 'fully_automated'
    phased: bool  # Whether expansion is phased or single-phase
    phase_duration: int = 3  # Duration in years if phased
    
    def get_total_cost(self) -> float:
        """Calculate total investment cost"""
        base_costs = {0.15: 120, 0.40: 450, 0.85: 750}  # Base costs in millions
        tech_multipliers = {'manual': 1.0, 'semi_automated': 1.2, 'fully_automated': 1.5}
        
        base_cost = base_costs.get(self.capacity_increase, 450)
        tech_multiplier = tech_multipliers.get(self.technology, 1.0)
        phasing_multiplier = 1.15 if self.phased else 1.0  # Phasing adds 15% cost
        
        return base_cost * tech_multiplier * phasing_multiplier

@dataclass 
class PortParameters:
    """Port operating parameters for cost calculations"""
    current_capacity: float = 3.2  # Million TEU
    current_demand: float = 2.8  # Million TEU
    revenue_per_teu: float = 180  # Dollars per TEU
    annual_growth_rate: float = 0.07  # 7% annual demand growth
    discount_rate: float = 0.08  # 8% discount rate
    planning_horizon: int = 15  # Years
    
class VariableNeighborhoodDescent:
    """Variable Neighborhood Descent for port capacity expansion"""
    
    def __init__(self, port_params: PortParameters):
        self.port_params = port_params
        self.neighborhood_functions = [
            self._neighborhood_timing,      # Neighborhood 1: Adjust timing
            self._neighborhood_capacity,    # Neighborhood 2: Change capacity level
            self._neighborhood_technology,   # Neighborhood 3: Change technology
            self._neighborhood_phasing       # Neighborhood 4: Toggle phasing
        ]
        self.iteration_history = []
        
    def _neighborhood_timing(self, solution: ExpansionSolution) -> ExpansionSolution:
        """Neighborhood 1: Adjust expansion timing"""
        neighbor = deepcopy(solution)
        
        # Adjust timing by ±1 year (within bounds)
        year_change = random.choice([-1, 1])
        neighbor.expansion_year = max(1, min(10, neighbor.expansion_year + year_change))
        
        return neighbor
    
    def _neighborhood_capacity(self, solution: ExpansionSolution) -> ExpansionSolution:
        """Neighborhood 2: Change capacity level"""
        neighbor = deepcopy(solution)
        
        # Cycle through capacity options
        capacity_options = [0.15, 0.40, 0.85]
        current_idx = capacity_options.index(neighbor.capacity_increase)
        new_idx = (current_idx + random.choice([-1, 1])) % len(capacity_options)
        neighbor.capacity_increase = capacity_options[new_idx]
        
        return neighbor
    
    def _neighborhood_technology(self, solution: ExpansionSolution) -> ExpansionSolution:
        """Neighborhood 3: Change technology level"""
        neighbor = deepcopy(solution)
        
        # Change to different technology
        tech_options = ['manual', 'semi_automated', 'fully_automated']
        current_tech = neighbor.technology
        neighbor.technology = random.choice([t for t in tech_options if t != current_tech])
        
        return neighbor
    
    def _neighborhood_phasing(self, solution: ExpansionSolution) -> ExpansionSolution:
        """Neighborhood 4: Toggle phasing and adjust duration"""
        neighbor = deepcopy(solution)
        
        # Toggle phasing
        neighbor.phased = not neighbor.phased
        
        # If now phased, set duration
        if neighbor.phased:
            neighbor.phase_duration = random.choice([2, 3, 4])
        
        return neighbor
    
    def evaluate_solution(self, solution: ExpansionSolution) -> float:
        """Evaluate solution quality (lower is better - negative NPV)"""
        # Calculate costs and benefits
        investment_cost = solution.get_total_cost()
        operational_benefit = self._calculate_operational_benefit(solution)
        congestion_savings = self._calculate_congestion_savings(solution)
        
        # Total NPV (negative cost, positive benefit)
        total_npv = operational_benefit + congestion_savings - investment_cost
        
        # Return negative for minimization (higher NPV = lower cost)
        return -total_npv
    
    def _calculate_operational_benefit(self, solution: ExpansionSolution) -> float:
        """Calculate operational benefits from expansion"""
        # Simplified benefit calculation
        years_to_expansion = solution.expansion_year
        new_capacity = self.port_params.current_capacity * (1 + solution.capacity_increase)
        
        # Technology efficiency gains
        tech_efficiency = {'manual': 1.0, 'semi_automated': 1.15, 'fully_automated': 1.30}
        efficiency_gain = tech_efficiency.get(solution.technology, 1.0)
        
        # Calculate additional revenue from increased capacity
        additional_capacity = new_capacity - self.port_params.current_capacity
        annual_revenue = additional_capacity * self.port_params.revenue_per_teu * efficiency_gain
        
        # Discount to present value
        present_value = 0
        for year in range(years_to_expansion, self.port_params.planning_horizon):
            discounted_revenue = annual_revenue / ((1 + self.port_params.discount_rate) ** year)
            present_value += discounted_revenue
        
        return present_value
    
    def _calculate_congestion_savings(self, solution: ExpansionSolution) -> float:
        """Calculate congestion cost savings"""
        years_to_expansion = solution.expansion_year
        new_capacity = self.port_params.current_capacity * (1 + solution.capacity_increase)
        
        # Current and future utilization
        current_utilization = self.port_params.current_demand / self.port_params.current_capacity
        future_demand = self.port_params.current_demand * ((1 + self.port_params.annual_growth_rate) ** years_to_expansion)
        future_utilization_without_expansion = min(future_demand / self.port_params.current_capacity, 1.0)
        future_utilization_with_expansion = min(future_demand / new_capacity, 1.0)
        
        # Congestion penalty function (exponential above 80% utilization)
        def congestion_penalty(utilization):
            if utilization <= 0.8:
                return 0
            else:
                return 1000 * np.exp(2 * (utilization - 0.8)) * 365  # Annual penalty
        
        # Annual savings
        annual_savings = (congestion_penalty(future_utilization_without_expansion) - 
                         congestion_penalty(future_utilization_with_expansion))
        
        # Discount to present value
        present_value = 0
        for year in range(years_to_expansion, self.port_params.planning_horizon):
            discounted_savings = annual_savings / ((1 + self.port_params.discount_rate) ** year)
            present_value += discounted_savings
        
        return present_value
    
    def solve(self, initial_solution: ExpansionSolution, max_iterations: int = 100) -> Tuple[ExpansionSolution, float, List]:
        """Solve using Variable Neighborhood Descent"""
        current_solution = deepcopy(initial_solution)
        current_cost = self.evaluate_solution(current_solution)
        
        self.iteration_history = [{
            'iteration': 0,
            'solution': deepcopy(current_solution),
            'cost': current_cost,
            'neighborhood': -1,
            'improvement': 0
        }]
        
        iteration = 0
        improved = True
        
        while improved and iteration < max_iterations:
            improved = False
            
            for k, neighborhood_func in enumerate(self.neighborhood_functions):
                # Generate neighbor in k-th neighborhood
                neighbor_solution = neighborhood_func(current_solution)
                neighbor_cost = self.evaluate_solution(neighbor_solution)
                
                # Accept if better (lower cost)
                if neighbor_cost < current_cost:
                    improvement = current_cost - neighbor_cost
                    current_solution = neighbor_solution
                    current_cost = neighbor_cost
                    improved = True
                    
                    iteration += 1
                    self.iteration_history.append({
                        'iteration': iteration,
                        'solution': deepcopy(current_solution),
                        'cost': current_cost,
                        'neighborhood': k + 1,
                        'improvement': improvement
                    })
                    
                    break  # Restart from first neighborhood
        
        return current_solution, -current_cost, self.iteration_history  # Return positive NPV

print("VND algorithm classes defined successfully")

In [ ]:
# Initialize port parameters and create initial solution
port_params = PortParameters()

# Create initial solution (from source example)
initial_solution = ExpansionSolution(
    expansion_year=6,  # 2026 (6 years from 2020 baseline)
    capacity_increase=0.40,  # 40% increase
    technology='semi_automated',
    phased=False,
    phase_duration=3
)

print("INITIAL SOLUTION CONFIGURATION:")
print(f"Expansion Year: {2020 + initial_solution.expansion_year}")
print(f"Capacity Increase: {initial_solution.capacity_increase:.0%}")
print(f"Technology: {initial_solution.expansion_year}")
print(f"Phased: {initial_solution.phased}")
print(f"Total Cost: ${initial_solution.get_total_cost():.1f}M")

# Create VND solver
vnd_solver = VariableNeighborhoodDescent(port_params)

# Evaluate initial solution
initial_cost = vnd_solver.evaluate_solution(initial_solution)
initial_npv = -initial_cost

print(f"\nInitial Solution NPV: ${initial_npv:.1f}M")
print(f"Initial Cost (for minimization): {initial_cost:.1f}")

In [ ]:
# Run Variable Neighborhood Descent
print("\n" + "="*60)
print("RUNNING VARIABLE NEIGHBORHOOD DESCENT OPTIMIZATION")
print("="*60)

final_solution, final_npv, iteration_history = vnd_solver.solve(initial_solution, max_iterations=50)

print(f"\nOptimization completed in {len(iteration_history)-1} iterations")
print(f"Final NPV: ${final_npv:.1f}M")
print(f"Improvement: ${final_npv - initial_npv:.1f}M ({((final_npv - initial_npv)/abs(initial_npv))*100:.1f}%)")

print("\n" + "="*60)
print("FINAL OPTIMIZED SOLUTION")
print("="*60)
print(f"Expansion Year: {2020 + final_solution.expansion_year}")
print(f"Capacity Increase: {final_solution.capacity_increase:.0%}")
print(f"Technology: {final_solution.technology}")
print(f"Phased: {final_solution.phased}")
if final_solution.phased:
    print(f"Phase Duration: {final_solution.phase_duration} years")
print(f"Total Cost: ${final_solution.get_total_cost():.1f}M")
print(f"Final NPV: ${final_npv:.1f}M")

# Show neighborhood improvements
print("\n" + "="*60)
print("NEIGHBORHOOD IMPROVEMENT SUMMARY")
print("="*60)

neighborhood_names = ["Timing", "Capacity", "Technology", "Phasing"]
neighborhood_improvements = {name: 0 for name in neighborhood_names}
neighborhood_count = {name: 0 for name in neighborhood_names}

for record in iteration_history[1:]:  # Skip initial record
    if record['improvement'] > 0:
        neighborhood_idx = record['neighborhood'] - 1
        neighborhood_name = neighborhood_names[neighborhood_idx]
        neighborhood_improvements[neighborhood_name] += record['improvement']
        neighborhood_count[neighborhood_name] += 1

for name in neighborhood_names:
    if neighborhood_count[name] > 0:
        avg_improvement = neighborhood_improvements[name] / neighborhood_count[name]
        print(f"{name}: {neighborhood_count[name]} improvements, avg ${avg_improvement:.1f}M")
    else:
        print(f"{name}: No improvements")

In [ ]:
# Detailed iteration analysis
print("\n" + "="*80)
print("DETAILED ITERATION ANALYSIS")
print("="*80)

# Create detailed analysis table
analysis_data = []
for i, record in enumerate(iteration_history):
    if i == 0:
        analysis_data.append({
            'Iteration': record['iteration'],
            'Neighborhood': 'Initial',
            'Year': 2020 + record['solution'].expansion_year,
            'Capacity': f"{record['solution'].capacity_increase:.0%}",
            'Technology': record['solution'].technology,
            'Phased': record['solution'].phased,
            'Cost ($M)': record['solution'].get_total_cost(),
            'NPV ($M)': -record['cost'],
            'Improvement ($M)': 0
        })
    else:
        neighborhood_name = neighborhood_names[record['neighborhood'] - 1]
        analysis_data.append({
            'Iteration': record['iteration'],
            'Neighborhood': neighborhood_name,
            'Year': 2020 + record['solution'].expansion_year,
            'Capacity': f"{record['solution'].capacity_increase:.0%}",
            'Technology': record['solution'].technology,
            'Phased': record['solution'].phased,
            'Cost ($M)': record['solution'].get_total_cost(),
            'NPV ($M)': -record['cost'],
            'Improvement ($M)': record['improvement']
        })

analysis_df = pd.DataFrame(analysis_data)
display(analysis_df.head(10))  # Show first 10 iterations

if len(analysis_df) > 10:
    print(f"... and {len(analysis_df) - 10} more iterations")

In [ ]:
# Create comprehensive visualizations
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Variable Neighborhood Descent Optimization Results', fontsize=16, fontweight='bold')

# Extract data for plotting
iterations = [record['iteration'] for record in iteration_history]
npvs = [-record['cost'] for record in iteration_history]
neighborhoods = [record['neighborhood'] for record in iteration_history]
improvements = [record['improvement'] for record in iteration_history]

# 1. NPV Progress Over Iterations
axes[0, 0].plot(iterations, npvs, 'b-o', linewidth=2, markersize=6)
axes[0, 0].set_title('NPV Improvement Over Iterations')
axes[0, 0].set_xlabel('Iteration')
axes[0, 0].set_ylabel('NPV ($M)')
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].axhline(y=initial_npv, color='red', linestyle='--', alpha=0.7, label='Initial NPV')
axes[0, 0].legend()

# 2. Neighborhood Contribution
neighborhood_contributions = {}
for i, record in enumerate(iteration_history[1:], 1):
    if record['improvement'] > 0:
        neighborhood_name = neighborhood_names[record['neighborhood'] - 1]
        if neighborhood_name not in neighborhood_contributions:
            neighborhood_contributions[neighborhood_name] = []
        neighborhood_contributions[neighborhood_name].append(record['improvement'])

if neighborhood_contributions:
    neighborhoods_list = list(neighborhood_contributions.keys())
    contributions = [sum(neighborhood_contributions[name]) for name in neighborhoods_list]
    colors_list = ['skyblue', 'lightgreen', 'salmon', 'gold']
    
    bars = axes[0, 1].bar(neighborhoods_list, contributions, color=colors_list[:len(neighborhoods_list)])
    axes[0, 1].set_title('Total Improvement by Neighborhood')
    axes[0, 1].set_ylabel('Total NPV Improvement ($M)')
    axes[0, 1].tick_params(axis='x', rotation=45)
    
    # Add value labels
    for bar, contribution in zip(bars, contributions):
        axes[0, 1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                        f'${contribution:.1f}M', ha='center', va='bottom')
else:
    axes[0, 1].text(0.5, 0.5, 'No improvements found', ha='center', va='center', 
                    transform=axes[0, 1].transAxes, fontsize=12)
    axes[0, 1].set_title('Total Improvement by Neighborhood')

# 3. Solution Evolution - Capacity Changes
capacities = [record['solution'].capacity_increase for record in iteration_history]
unique_caps = list(set(capacities))
cap_colors = plt.cm.Set3(np.linspace(0, 1, len(unique_caps)))
cap_color_map = {cap: color for cap, color in zip(unique_caps, cap_colors)}

for i in range(len(iterations) - 1):
    axes[1, 0].plot(iterations[i:i+2], capacities[i:i+2], 
                     'o-', color=cap_color_map[capacities[i]], linewidth=2, markersize=8)

axes[1, 0].set_title('Capacity Level Evolution')
axes[1, 0].set_xlabel('Iteration')
axes[1, 0].set_ylabel('Capacity Increase (%)')
axes[1, 0].set_yticks([0.15, 0.40, 0.85])
axes[1, 0].set_yticklabels(['15%', '40%', '85%'])
axes[1, 0].grid(True, alpha=0.3)

# 4. Solution Evolution - Timing Changes
years = [2020 + record['solution'].expansion_year for record in iteration_history]
axes[1, 1].plot(iterations, years, 'g-o', linewidth=2, markersize=6)
axes[1, 1].set_title('Expansion Year Evolution')
axes[1, 1].set_xlabel('Iteration')
axes[1, 1].set_ylabel('Expansion Year')
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].axhline(y=2020 + initial_solution.expansion_year, color='red', 
                   linestyle='--', alpha=0.7, label='Initial Year')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

print("Visualization complete - VND optimization results displayed")

In [ ]:
# Compare initial vs final solution in detail
print("\n" + "="*80)
print("COMPREHENSIVE SOLUTION COMPARISON")
print("="*80)

def analyze_solution_detailed(solution: ExpansionSolution, name: str):
    """Perform detailed analysis of a solution"""
    cost = solution.get_total_cost()
    npv = -vnd_solver.evaluate_solution(solution)
    operational_benefit = vnd_solver._calculate_operational_benefit(solution)
    congestion_savings = vnd_solver._calculate_congestion_savings(solution)
    
    print(f"\n{name} SOLUTION ANALYSIS:")
    print(f"  Expansion Year: {2020 + solution.expansion_year}")
    print(f"  Capacity Increase: {solution.capacity_increase:.0%}")
    print(f"  Technology: {solution.technology}")
    print(f"  Phased: {solution.phased}")
    if solution.phased:
        print(f"  Phase Duration: {solution.phase_duration} years")
    print(f"  Investment Cost: ${cost:.1f}M")
    print(f"  Operational Benefit: ${operational_benefit:.1f}M")
    print(f"  Congestion Savings: ${congestion_savings:.1f}M")
    print(f"  Net Present Value: ${npv:.1f}M")
    
    return {
        'cost': cost,
        'npv': npv,
        'operational_benefit': operational_benefit,
        'congestion_savings': congestion_savings
    }

# Analyze both solutions
initial_analysis = analyze_solution_detailed(initial_solution, "INITIAL")
final_analysis = analyze_solution_detailed(final_solution, "FINAL")

# Calculate improvements
print("\n" + "="*60)
print("IMPROVEMENT ANALYSIS")
print("="*60)

npv_improvement = final_analysis['npv'] - initial_analysis['npv']
cost_change = final_analysis['cost'] - initial_analysis['cost']
operational_improvement = final_analysis['operational_benefit'] - initial_analysis['operational_benefit']
congestion_improvement = final_analysis['congestion_savings'] - initial_analysis['congestion_savings']

print(f"NPV Improvement: ${npv_improvement:.1f}M ({(npv_improvement/abs(initial_analysis['npv']))*100:.1f}%)")
print(f"Investment Cost Change: ${cost_change:.1f}M ({(cost_change/initial_analysis['cost'])*100:.1f}%)")
print(f"Operational Benefit Improvement: ${operational_improvement:.1f}M")
print(f"Congestion Savings Improvement: ${congestion_improvement:.1f}M")

# Key insights
print("\n" + "="*60)
print("KEY INSIGHTS & RECOMMENDATIONS")
print("="*60)

if final_solution.expansion_year > initial_solution.expansion_year:
    print("📅 TIMING: Delayed expansion reduces financing costs and allows better demand visibility")
elif final_solution.expansion_year < initial_solution.expansion_year:
    print("📅 TIMING: Earlier expansion addresses capacity constraints proactively")

if final_solution.capacity_increase > initial_solution.capacity_increase:
    print("📈 CAPACITY: Larger capacity increase provides long-term growth buffer")
elif final_solution.capacity_increase < initial_solution.capacity_increase:
    print("📈 CAPACITY: Smaller capacity increase reduces financial risk")

if final_solution.technology != initial_solution.technology:
    print(f"🔧 TECHNOLOGY: Switch to {final_solution.technology} improves operational efficiency")

if final_solution.phased != initial_solution.phased:
    if final_solution.phased:
        print("⚡ PHASING: Phased implementation reduces peak financing and improves cash flow")
    else:
        print("⚡ PHASING: Single-phase implementation accelerates benefits delivery")

print(f"\n💰 OVERALL: VND optimization achieved ${npv_improvement:.1f}M NPV improvement")
print(f"   This represents a {(npv_improvement/abs(initial_analysis['npv']))*100:.1f}% improvement over initial configuration")

print("\n" + "="*80)